# 03 — Model construction

**Task.** Construct and compare three retrieval models on the same document contract and validation queries.

1. Benchmark: unigram TF-IDF.
2. Phrase-aware model: unigram + bigram TF-IDF.
3. Semantic model: pretrained transformer embeddings.

The TF-IDF pipelines run in Spark ML. Spark prepares the transformer input and consumes its output, while the specialized neural-network inference step uses batched PyTorch/SentenceTransformers on the controlled sample. Exact retrieval routines are retained as controlled small-sample audits so LSH approximation does not confound representation comparison.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'final_project' else Path.cwd().resolve()
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from jobapps.pipelines.ngram_benchmark import run_ngram_benchmark
from jobapps.pipelines.transformer_benchmark import run_transformer_benchmark

QUALITY_PATH = PROJECT_ROOT / 'config' / 'data_quality.yaml'

## Models 1 and 2: Spark TF-IDF pipelines

The unigram benchmark applies `RegexTokenizer`, `StopWordsRemover`, `HashingTF`, `IDF`, and L2 normalization. The n-gram model adds Spark `NGram(n=2)` and combines prefixed unigram and bigram tokens before hashing. Both fit IDF on jobs only.

In [ ]:
ngram_metrics = run_ngram_benchmark(
    PROJECT_ROOT / 'config' / 'ngram_1pct.yaml',
    QUALITY_PATH,
)
ngram_metrics

## Model 3: transformer embeddings

Spark prepares the typed validation documents. A single MiniLM model then creates overlapping tokenizer-aware chunks, performs batched PyTorch inference, mean-pools chunks, and normalizes document vectors. The vectors return to Spark for Parquet persistence and evaluation. This boundary was chosen after two local Spark-worker trials failed during Arrow transfer despite safe memory use; the direct PyTorch run completed successfully. Cached embeddings prevent repeated inference.

In [ ]:
transformer_metrics = run_transformer_benchmark(
    PROJECT_ROOT / 'config' / 'transformer_1pct.yaml',
    QUALITY_PATH,
)
transformer_metrics

These calls use the validation split. Do not change `resume_query_split` to `test` until the feature design and retrieval parameters are frozen.